## Build a machine learning model that classifies news as REAL or FAKE using Natural Language Processing (NLP).

### Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Step 2 — Load Dataset

In [2]:
fake = pd.read_csv(r"C:/Users/mansi chauhan/ML_Project/datasets/Fake.csv")
true = pd.read_csv(r"C:/Users/mansi chauhan/ML_Project/datasets/True.csv")

In [3]:
fake.head()
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


### Step 3 — Add Labels

In [4]:
fake["label"] = 0
true["label"] = 1

### Step 4 — Combine Dataset

In [5]:
data = pd.concat([fake, true], axis=0)

data = data.sample(frac=1).reset_index(drop=True)

data.head()

,title,text,subject,date,label
0,Boiler Room #108 – Who’d Win in a Fight? Boile...,Tune in to the Alternate Current Radio Network...,Middle-east,"May 12, 2017",0
1,Conservatives Demand Purge Of LGBT Employees ...,Conservatives apparently want a witch hunt in ...,News,"December 18, 2016",0
2,Carson cancels campaign events after staff in ...,WASHINGTON (Reuters) - Republican presidential...,politicsNews,"January 19, 2016",1
3,Lebanon's Hariri says to hold off resignation ...,BEIRUT (Reuters) - Lebanon s Saad al-Hariri sa...,worldnews,"November 22, 2017",1
4,WOW! THIS LETTER FROM JULY 30TH PREDICTS OBAMA...,Obama s war against America on every possible ...,politics,"Aug 14, 2015",0


### Step 5 — Check Dataset Info

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [7]:
data["label"].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

### Step 6 — Select Required Columns

In [8]:
data["content"] = data["title"] + " " + data["text"]

data = data[["content", "label"]]

data.head()

,content,label
0,Boiler Room #108 – Who’d Win in a Fight? Boile...,0
1,Conservatives Demand Purge Of LGBT Employees ...,0
2,Carson cancels campaign events after staff in ...,1
3,Lebanon's Hariri says to hold off resignation ...,1
4,WOW! THIS LETTER FROM JULY 30TH PREDICTS OBAMA...,0


### Step 7 — Text Cleaning Function

In [9]:
def clean_text(text):
    
    text = text.lower()
    
    text = re.sub(r'\[.*?\]', '', text)
    
    text = re.sub(r"\W"," ",text)
    
    text = re.sub(r"\d"," ",text)
    
    text = re.sub(r"\s+"," ",text)
    
    return text

In [10]:
data["content"] = data["content"].apply(clean_text)

### Step 8 — Define Features and Labels

In [11]:
X = data["content"]
y = data["label"]

### Step 9 — Train Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Step 10 — Convert Text to Numbers (TF-IDF)

In [13]:
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### Step 11 — Train Machine Learning Model

In [14]:
model = LogisticRegression()

model.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


### Step 12 — Make Predictions

In [15]:
y_pred = model.predict(X_test_vec)

### Step 13 — Evaluate Model

In [16]:
# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.984966592427617


In [17]:
# Classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4739
           1       0.98      0.98      0.98      4241

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [18]:
# Confusion matrix
print(confusion_matrix(y_test, y_pred))

[[4674   65]
 [  70 4171]]


### Step 14 — Create Fake News Prediction Function

In [19]:
def predict_news(news):

    news = clean_text(news)
    
    vector = vectorizer.transform([news])
    
    prediction = model.predict(vector)
    
    probability = model.predict_proba(vector)
    
    if prediction[0] == 1:
        print("Real News")
        print("Confidence:", probability[0][1])
    else:
        print("Fake News")
        print("Confidence:", probability[0][0])

### Step 15 — Test the Model

In [20]:
predict_news("Breaking: Government announces new policy for education sector")

Fake News
Confidence: 0.8701865140376024


In [21]:
predict_news("Scientists say miracle fruit can cure all diseases overnight")

Fake News
Confidence: 0.672620165713065


### Step 16 — Save Model

In [22]:
import pickle

pickle.dump(model, open("fake_news_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))

### Step 17 — Load Model Later

In [23]:
model = pickle.load(open("fake_news_model.pkl","rb"))
vectorizer = pickle.load(open("vectorizer.pkl","rb"))